In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
抓取软科中国大学排名（主榜）中“排名”和“学校名称”，并导出到 Excel。

默认目标页：
https://www.shanghairanking.cn/rankings/bcur/2026

用法：
    pip install requests openpyxl
    python shanghairanking_bcur_2026_to_excel.py

也可以指定 URL 或输出文件名：
    python shanghairanking_bcur_2026_to_excel.py \
        --url https://www.shanghairanking.cn/rankings/bcur/2026 \
        --output 软科中国大学排名_2026.xlsx
"""

from __future__ import annotations

import argparse
import re
from pathlib import Path
from typing import Dict, List, Tuple

import requests
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DEFAULT_PAGE_URL = "https://www.shanghairanking.cn/rankings/bcur/2026"
DEFAULT_BCUR_TYPE = 11


def extract_year_from_url(url: str) -> str:
    """从页面 URL 中提取年份，例如 /bcur/2026 -> 2026。"""
    match = re.search(r"/bcur/(\d{4})(?:/)?$", url)
    if not match:
        raise ValueError(f"无法从 URL 中解析年份：{url}")
    return match.group(1)


def build_session() -> requests.Session:
    """构建带重试机制的 requests 会话。"""
    session = requests.Session()

    retry = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    session.headers.update(
        {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
            "Accept": "application/json, text/plain, */*",
            "Referer": DEFAULT_PAGE_URL,
        }
    )
    return session


def fetch_rankings(year: str, bcur_type: int = DEFAULT_BCUR_TYPE) -> Tuple[List[Dict[str, str]], str]:
    """
    调用接口获取排名数据。

    返回：
        rows: [{"排名": "1", "学校名称": "清华大学"}, ...]
        api_url: 实际请求的接口地址
    """
    api_url = f"https://www.shanghairanking.cn/api/pub/v1/bcur?bcur_type={bcur_type}&year={year}"
    session = build_session()
    response = session.get(api_url, timeout=20)
    response.raise_for_status()

    try:
        payload = response.json()
    except ValueError as exc:
        raise RuntimeError("接口返回不是合法 JSON，站点接口可能已改版。") from exc

    code = payload.get("code")
    if code not in (200, "200", None):
        raise RuntimeError(f"接口返回异常：code={code}，msg={payload.get('msg')}")

    rankings = (payload.get("data") or {}).get("rankings") or []
    if not rankings:
        raise RuntimeError("未拿到 rankings 数据，站点接口可能已改版。")

    rows: List[Dict[str, str]] = []
    seen = set()

    for item in rankings:
        rank = str(item.get("ranking") or item.get("rankOverall") or "").strip()
        school_name = str(item.get("univNameCn") or "").strip()

        if not school_name:
            continue

        key = (rank, school_name)
        if key in seen:
            continue
        seen.add(key)

        rows.append({"排名": rank, "学校名称": school_name})

    if not rows:
        raise RuntimeError("解析后为空，站点字段可能已改版。")

    return rows, api_url


def save_to_excel(rows: List[Dict[str, str]], output_file: Path, sheet_name: str) -> None:
    """把结果保存到 Excel，并做一个简洁可读的表格样式。"""
    wb = Workbook()
    ws = wb.active
    ws.title = sheet_name

    ws.append(["排名", "学校名称"])
    for row in rows:
        ws.append([row["排名"], row["学校名称"]])

    header_fill = PatternFill(fill_type="solid", fgColor="1F4E78")
    alt_fill = PatternFill(fill_type="solid", fgColor="F7FBFF")
    header_font = Font(color="FFFFFF", bold=True)
    normal_font = Font(color="000000")

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:B{ws.max_row}"
    ws.column_dimensions["A"].width = 12
    ws.column_dimensions["B"].width = 36

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")

    for row_idx in range(2, ws.max_row + 1):
        rank_cell = ws.cell(row=row_idx, column=1)
        school_cell = ws.cell(row=row_idx, column=2)

        rank_cell.font = normal_font
        school_cell.font = normal_font

        rank_cell.alignment = Alignment(horizontal="center", vertical="center")
        school_cell.alignment = Alignment(horizontal="left", vertical="center")

        if row_idx % 2 == 0:
            rank_cell.fill = alt_fill
            school_cell.fill = alt_fill

    wb.save(output_file)


def main() -> None:
    parser = argparse.ArgumentParser(description="抓取软科中国大学排名并导出 Excel")
    parser.add_argument(
        "--url",
        default=DEFAULT_PAGE_URL,
        help="软科排名页面 URL，默认是 2026 主榜页面",
    )
    parser.add_argument(
        "--output",
        default=None,
        help="输出的 Excel 文件名，默认：软科中国大学排名_年份.xlsx",
    )
    parser.add_argument(
        "--bcur-type",
        type=int,
        default=DEFAULT_BCUR_TYPE,
        help="榜单类型参数，主榜默认 11",
    )
    args, _ = parser.parse_known_args()

    year = extract_year_from_url(args.url)
    output_file = Path(args.output or f"软科中国大学排名_{year}.xlsx")
    sheet_name = f"{year}中国大学排名"

    rows, api_url = fetch_rankings(year=year, bcur_type=args.bcur_type)
    save_to_excel(rows, output_file=output_file, sheet_name=sheet_name)

    print(f"抓取完成，共 {len(rows)} 条记录")
    print(f"输出文件：{output_file.resolve()}")
    print(f"使用接口：{api_url}")


if __name__ == "__main__":
    main()


抓取完成，共 590 条记录
输出文件：/Users/bangongyi/Desktop/repengwu/迁移/旧电脑桌面/招生计划标签提取脚本/软科中国大学排名_2026.xlsx
使用接口：https://www.shanghairanking.cn/api/pub/v1/bcur?bcur_type=11&year=2026


In [2]:
pip install playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 3.5 MB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [playwright]3 [playwright]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import sys
print(sys.executable)

!{sys.executable} -m pip install playwright pandas openpyxl nest_asyncio -q
!{sys.executable} -m playwright install chromium

/usr/local/Caskroom/miniconda/base/bin/python

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
(node:62376) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
170.9 MiB [                    ] 0% 0.0s170.9 MiB [                    ] 0% 322.8s170.9 MiB [                    ] 0% 1481.6s170.9 MiB [                    ] 0% 1247.5s170.9 MiB [                    ] 0% 1158.4s170.9 MiB [                    ] 0% 1087.0s170.9 MiB [                    ] 0% 1035.5s170.9 MiB [                    ] 0% 988.6s170.9 MiB [                    ] 0% 954.7s170.9 MiB [                    ] 0% 984.0s170.9 MiB [                    ] 0% 908.9s170.9 MiB [                    ] 0% 860.8s170.9 MiB [                    ] 0%

In [7]:
!{sys.executable} -m playwright install chrome


╔═════════════════════════════════════════════════════════════════╗
║ ATTENTION: "chrome" is already installed on the system!         ║
║                                                                 ║
║ "chrome" installation is not hermetic; installing newer version ║
║ requires *removal* of a current installation first.             ║
║                                                                 ║
║ To *uninstall* current version and re-install latest "chrome":  ║
║                                                                 ║
║ - Close all running instances of "chrome", if any               ║
║ - Use "--force" to install browser:                             ║
║                                                                 ║
║     playwright install --force chrome                           ║
║                                                                 ║
║ <3 Playwright Team                                              ║
╚══════════════════════════════════════════════

In [12]:
import sys
print(sys.executable)

!{sys.executable} -m playwright install chromium

/usr/local/Caskroom/miniconda/base/bin/python


In [13]:
import asyncio
import json
import re
import time
import getpass
from pathlib import Path
from typing import Any, Dict, List, Optional

import nest_asyncio
import pandas as pd
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

nest_asyncio.apply()


# =========================================================
# 基础配置
# =========================================================

TARGET_URL = "https://www.baodh.cn/b/Nmg81MakeFaNewPage.do?m=MW"

USERNAME = "8112503"
PASSWORD = "565506"

MIN_SCORE = 100
MAX_SCORE = 700

OUT_PREFIX = "baodh_nm_majors_100_700"

GROUP_LINK_PATTERN = re.compile(r"^\s*\d+\s*个专业\s*$")

API_KEYWORDS = (
    "Nmg81_QueryMajorLstByCollege",
    "Nmg81_QueryMajorLstByColl",
    "Nmg81_XGK_QueryCollegeZy",
    "Nmg81_Xgk_QueryCollegeZy",
    "Nmg81_QueryFaZyzMajor",
    "QueryMajor",
    "CollegeZy",
    "MajorLst",
    "ZyzMajor",
)


# =========================================================
# 工具函数
# =========================================================

def now_str() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()


def extract_list_from_json(data: Any) -> List[Dict[str, Any]]:
    """
    从接口 JSON 中尽量提取列表数据。
    """
    if isinstance(data, list):
        return [x for x in data if isinstance(x, dict)]

    if isinstance(data, dict):
        for key in ("lst", "list", "rows", "data", "result", "items"):
            val = data.get(key)

            if isinstance(val, list):
                return [x for x in val if isinstance(x, dict)]

            if isinstance(val, dict):
                inner = extract_list_from_json(val)
                if inner:
                    return inner

        for val in data.values():
            if isinstance(val, dict):
                inner = extract_list_from_json(val)
                if inner:
                    return inner

    return []


def flatten_dict(obj: Dict[str, Any], prefix: str = "") -> Dict[str, Any]:
    """
    把嵌套 JSON 打平成一行。
    """
    out = {}

    for k, v in obj.items():
        key = f"{prefix}.{k}" if prefix else str(k)

        if isinstance(v, dict):
            out.update(flatten_dict(v, key))
        elif isinstance(v, list):
            out[key] = json.dumps(v, ensure_ascii=False)
        else:
            out[key] = v

    return out


def parse_group_text(group_text: str) -> Dict[str, str]:
    """
    从页面文字里尽量解析院校代码、院校名称、专业组。
    """
    text = normalize_text(group_text)

    result = {
        "院校代码_页面解析": "",
        "院校名称_页面解析": "",
        "专业组_页面解析": "",
    }

    m = re.search(r"\(([^)]+)\)\s*([^\s]+)", text)
    if m:
        result["院校代码_页面解析"] = m.group(1).strip()
        result["院校名称_页面解析"] = m.group(2).strip()

    g = re.search(r"第\s*([0-9A-Za-z]+)\s*组", text)
    if g:
        result["专业组_页面解析"] = f"第{g.group(1)}组"

    return result


def save_outputs(rows: List[Dict[str, Any]], raw_items: List[Dict[str, Any]], out_prefix: str) -> pd.DataFrame:
    """
    保存 CSV、Excel、原始 JSONL。
    """
    raw_path = Path(f"{out_prefix}_raw.jsonl").resolve()
    csv_path = Path(f"{out_prefix}.csv").resolve()
    xlsx_path = Path(f"{out_prefix}.xlsx").resolve()

    with raw_path.open("w", encoding="utf-8") as f:
        for item in raw_items:
            item_to_save = {
                "url": item.get("url"),
                "status": item.get("status"),
                "row_count": item.get("row_count"),
                "data": item.get("data"),
            }
            f.write(json.dumps(item_to_save, ensure_ascii=False) + "\n")

    if not rows:
        print(f"没有抓到专业明细数据，只保存了原始响应：{raw_path}")
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    front_cols = [
        "抓取时间",
        "分数下限",
        "分数上限",
        "专业组序号",
        "专业组链接文本",
        "院校代码_页面解析",
        "院校名称_页面解析",
        "专业组_页面解析",
        "专业组页面文本",
        "明细接口",
    ]

    cols = [c for c in front_cols if c in df.columns] + [
        c for c in df.columns if c not in front_cols
    ]

    df = df[cols]

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df.to_excel(xlsx_path, index=False)

    print("保存完成：")
    print(f"CSV：{csv_path}")
    print(f"Excel：{xlsx_path}")
    print(f"原始 JSONL：{raw_path}")
    print(f"总行数：{len(df)}")

    return df


# =========================================================
# 接口抓包收集器
# =========================================================

class XhrCollector:
    def __init__(self):
        self.items = []

    async def handle_response(self, response):
        url = response.url

        if not any(k in url for k in API_KEYWORDS):
            return

        try:
            data = await response.json()
        except Exception:
            try:
                text = await response.text()
                data = json.loads(text)
            except Exception:
                return

        rows = extract_list_from_json(data)

        self.items.append(
            {
                "ts": time.time(),
                "url": url,
                "status": response.status,
                "data": data,
                "rows": rows,
                "row_count": len(rows),
            }
        )


# =========================================================
# 页面操作函数
# =========================================================

async def first_visible_locator(page, selectors):
    for sel in selectors:
        try:
            loc = page.locator(sel).first
            if await loc.count() > 0 and await loc.is_visible(timeout=1000):
                return loc
        except Exception:
            continue
    return None


async def auto_login(page, username: str, password: str):
    """
    自动填写账号密码并点击登录。
    如果识别不到登录框，就需要你手动登录。
    """
    username_selectors = [
        'input[name="username"]',
        'input[name="userName"]',
        'input[name="loginName"]',
        'input[name="account"]',
        'input[name="userid"]',
        'input[name="userId"]',
        'input[type="text"]',
        'input[placeholder*="账号"]',
        'input[placeholder*="用户名"]',
        'input[placeholder*="手机号"]',
        'input[placeholder*="考生"]',
    ]

    password_selectors = [
        'input[name="password"]',
        'input[name="pwd"]',
        'input[type="password"]',
        'input[placeholder*="密码"]',
    ]

    login_button_selectors = [
        'button:has-text("登录")',
        'input[type="button"][value*="登录"]',
        'input[type="submit"][value*="登录"]',
        'a:has-text("登录")',
        'div:has-text("登录")',
        'span:has-text("登录")',
    ]

    await page.wait_for_timeout(1500)

    user_box = await first_visible_locator(page, username_selectors)
    pwd_box = await first_visible_locator(page, password_selectors)

    if user_box is None or pwd_box is None:
        print("没有自动找到登录框。请你在浏览器里手动输入账号密码登录。")
        return False

    await user_box.fill(username)
    await pwd_box.fill(password)

    print("账号密码已填写。")

    login_btn = await first_visible_locator(page, login_button_selectors)

    if login_btn is not None:
        await login_btn.click()
        print("已点击登录按钮。")
    else:
        print("没有自动找到登录按钮，请你手动点击登录。")
        return False

    await page.wait_for_timeout(3000)
    return True


async def get_group_text_near_link(page, link_locator) -> str:
    """
    从“X个专业”链接向上找父级文本。
    """
    try:
        handle = await link_locator.element_handle(timeout=3000)

        if handle is None:
            return ""

        js = """
        (el) => {
            function norm(s) {
                return (s || '').replace(/\\s+/g, ' ').trim();
            }

            let p = el;

            for (let i = 0; i < 14 && p; i++, p = p.parentElement) {
                let t = norm(p.innerText || '');

                if (
                    t.length > 30 &&
                    (
                        t.includes('最高分专业') ||
                        t.includes('最低分专业') ||
                        t.includes('综合排名') ||
                        t.includes('专业组') ||
                        t.includes('招生人数') ||
                        t.includes('收藏')
                    )
                ) {
                    return t;
                }
            }

            return norm(el.innerText || '');
        }
        """

        return normalize_text(await page.evaluate(js, handle))

    except Exception:
        return ""


async def choose_best_new_response(collector: XhrCollector, start_index: int, wait_seconds: float = 8.0):
    """
    点击专业组后，从新捕获的接口响应里选择最像专业明细的 JSON。
    """
    deadline = time.time() + wait_seconds

    while time.time() < deadline:
        new_items = collector.items[start_index:]
        candidates = []

        for item in new_items:
            if item.get("status") != 200:
                continue

            rows = item.get("rows") or []

            if not rows:
                continue

            url = item.get("url", "")
            score = len(rows)

            priority_words = [
                "QueryMajorLstByCollege",
                "QueryMajorLstByColl",
                "QueryFaZyzMajor",
                "MajorLst",
                "ZyzMajor",
            ]

            if any(w in url for w in priority_words):
                score += 100000

            candidates.append((score, item))

        if candidates:
            candidates.sort(key=lambda x: x[0], reverse=True)
            return candidates[0][1]

        await asyncio.sleep(0.25)

    return None


async def load_all_group_links(page, max_scroll_rounds: int = 80) -> int:
    """
    先滚动页面，让懒加载的所有“X个专业”链接加载出来。
    """
    last_count = -1
    stable_rounds = 0

    for i in range(max_scroll_rounds):
        locator = page.get_by_role("link", name=GROUP_LINK_PATTERN)

        try:
            count = await locator.count()
        except Exception:
            count = 0

        if count == last_count:
            stable_rounds += 1
        else:
            stable_rounds = 0
            last_count = count

        print(f"滚动加载中：第 {i + 1} 次，当前检测到 {count} 个专业组链接", end="\r")

        await page.mouse.wheel(0, 3500)
        await page.wait_for_timeout(500)

        if stable_rounds >= 6:
            break

    print()

    await page.evaluate("window.scrollTo(0, 0)")
    await page.wait_for_timeout(1000)

    try:
        return await page.get_by_role("link", name=GROUP_LINK_PATTERN).count()
    except Exception:
        return 0


# =========================================================
# 主流程
# =========================================================

async def open_browser_and_login(username: str = USERNAME, password: str = PASSWORD):
    """
    打开浏览器并登录。
    """
    global playwright, browser, browser_context, page, collector

    try:
        await browser_context.close()
    except Exception:
        pass

    try:
        await browser.close()
    except Exception:
        pass

    try:
        await playwright.stop()
    except Exception:
        pass

    playwright = await async_playwright().start()

    try:
        browser = await playwright.chromium.launch(
            channel="chrome",
            headless=False,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-first-run",
                "--no-default-browser-check",
            ],
        )
    except Exception as e:
        print("系统 Chrome 启动失败，改用 Playwright Chromium。")
        print("错误信息：", repr(e))

        browser = await playwright.chromium.launch(
            headless=False,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-first-run",
                "--no-default-browser-check",
            ],
        )

    browser_context = await browser.new_context(
        viewport={"width": 1500, "height": 950}
    )

    page = await browser_context.new_page()

    collector = XhrCollector()

    page.on(
        "response",
        lambda response: asyncio.create_task(collector.handle_response(response))
    )

    await page.goto(TARGET_URL, wait_until="domcontentloaded", timeout=60000)

    print("页面已打开，准备登录。")

    ok = await auto_login(page, username, password)

    if ok:
        print("已尝试自动登录。")
    else:
        print("请在浏览器中手动登录。")

    print()
    print("登录后请在浏览器里手动完成筛选：")
    print(f"1. 设置分数区间：{MIN_SCORE} 到 {MAX_SCORE}")
    print("2. 设置科类、批次、计划性质等你需要的条件")
    print("3. 点击页面上的“确定”")
    print("4. 等学校 / 专业组列表加载完成")
    print()
    print("筛选完成后运行：")
    print("df_result = await crawl_all_major_groups()")


async def crawl_all_major_groups(
    min_score: int = MIN_SCORE,
    max_score: int = MAX_SCORE,
    out_prefix: str = OUT_PREFIX,
):
    """
    爬取当前筛选条件下所有学校、专业组、专业明细。
    """
    global final_rows, df_result

    final_rows = []

    total_links = await load_all_group_links(page)

    print(f"检测到专业组链接数量：{total_links}")

    if total_links == 0:
        raise RuntimeError("没有找到“X个专业”链接。请确认页面已经登录、筛选，并点击了确定。")

    visited_keys = set()
    processed = 0
    failed = 0

    for idx in range(total_links):
        locator = page.get_by_role("link", name=GROUP_LINK_PATTERN)

        try:
            current_count = await locator.count()

            if idx >= current_count:
                break

            link = locator.nth(idx)
            link_text = normalize_text(await link.inner_text(timeout=3000))

            group_text = await get_group_text_near_link(page, link)
            group_info = parse_group_text(group_text)

            unique_key = normalize_text(group_text)[:500] or f"{idx}-{link_text}"

            if unique_key in visited_keys:
                continue

            visited_keys.add(unique_key)

            print(f"\n{idx + 1}/{total_links} 点击：{link_text}")

            if group_info["院校名称_页面解析"] or group_info["专业组_页面解析"]:
                print(
                    f"{group_info['院校代码_页面解析']} "
                    f"{group_info['院校名称_页面解析']} "
                    f"{group_info['专业组_页面解析']}"
                )

            await link.scroll_into_view_if_needed(timeout=10000)
            await page.wait_for_timeout(300)

            before = len(collector.items)

            try:
                await link.click(timeout=10000)
            except PlaywrightTimeoutError:
                print("点击超时，跳过")
                failed += 1
                continue

            await page.wait_for_timeout(500)

            item = await choose_best_new_response(
                collector=collector,
                start_index=before,
                wait_seconds=8.0,
            )

            if item is None:
                print("没有捕获到专业明细 JSON，可能该组已展开、接口无数据或接口名未匹配。")
                failed += 1
                continue

            major_rows = item.get("rows") or []

            print(f"抓到专业数：{len(major_rows)}")

            for major in major_rows:
                row = {
                    "抓取时间": now_str(),
                    "分数下限": min_score,
                    "分数上限": max_score,
                    "专业组序号": idx + 1,
                    "专业组链接文本": link_text,
                    "专业组页面文本": group_text,
                    "明细接口": item["url"],
                }

                row.update(group_info)
                row.update(flatten_dict(major))

                final_rows.append(row)

            processed += 1

            await asyncio.sleep(0.4)

        except Exception as e:
            failed += 1
            print(f"第 {idx + 1} 个专业组处理失败：{type(e).__name__}: {e}")
            continue

    print()
    print("========== 抓取完成 ==========")
    print(f"成功处理专业组：{processed}")
    print(f"失败或未捕获专业组：{failed}")
    print(f"专业明细总行数：{len(final_rows)}")

    df_result = save_outputs(
        rows=final_rows,
        raw_items=collector.items,
        out_prefix=out_prefix,
    )

    return df_result


async def close_browser():
    """
    关闭浏览器。
    """
    try:
        await browser_context.close()
    except Exception:
        pass

    try:
        await browser.close()
    except Exception:
        pass

    try:
        await playwright.stop()
    except Exception:
        pass

    print("浏览器已关闭。")


print("完整代码已加载完成。")
print("下一步运行：await open_browser_and_login()")

完整代码已加载完成。
下一步运行：await open_browser_and_login()
